In [ ]:
%%bash
# # split UHVDB genomovars reps into chunks with 1K sequences
# seqkit split2 \
#     ../uhvdb_final_files/r2025_09/uhvdb_genomovars_reps.fna.gz \
#     --by-size 1000 \
#     --out-dir ./uhvdb_genomovar_chunks/

In [7]:
# create config file for annotations
import glob

task_id = 0
with open("annotations_config.txt", "w") as f:
    f.write("TaskArrayID\tFastA\tName\n")
    for file in glob.glob("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_4/uhvdb_genomovar_chunks/*.fna.gz"):
            f.write(f"{task_id}\t{file}\t{file.split('/')[-1].rsplit('.fna')[0]}\n")
            task_id += 1
f.close()

In [ ]:
# set up sbatch script for running bacphlip on all sequences
# run bacphlip

In [ ]:
%%bash
for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_4/uhvdb_genomovar_chunks/uhvdb_genomovars_reps.part_*.fna.bacphlip; do
    if $(grep -c "^") == 0; then
        echo $file
    fi
done

In [ ]:
# identify r2025_09 genomovars missing bacphlip data
import glob
import polars as pl

# load bacphlip results for UHVDB
bacphlip_lst = []
for file in glob.glob('/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_4/uhvdb_genomovar_chunks/uhvdb_genomovars_reps.part_*.fna.bacphlip'):
    df = (
        pl.read_csv(file, separator='\t', null_values=['NA'], new_columns=['contig_id', 'virulent', 'temperate'])
    )
    bacphlip_lst.append(df)

bacphlip_df = (
    pl.concat(bacphlip_lst)
)
bacphlip_df.height

432321

In [ ]:
# identify r2025_10 genomovars missing bacphlip data
import glob
import polars as pl

# load r2025_10 genomovar reps
r2025_10_clust = (
    pl.read_csv('/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_clustering_thru_votus.tsv', separator='\t')
    .unique('genomovar_rep')
)
print("Number of r2025_10 genomovars:", r2025_10_clust.height)

# load r2025_09 genomovar reps
r2025_09_clust = (
    pl.read_csv('/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_clustering.tsv.gz', separator='\t')
    .unique('genomovar_rep')
)
print("Number of r2025_09 genomovars:", r2025_09_clust.height)

# identify r2025_10 genomovars not in r2025_09
r2025_10_not_in_r2025_09 = (
    r2025_10_clust.filter(~pl.col('genomovar_rep').is_in(r2025_09_clust['genomovar_rep']))
)
print("Number of r2025_10 genomovars not in r2025_09:", r2025_10_not_in_r2025_09.height)

r2025_10_not_in_r2025_09[['genomovar_rep']].write_csv('r2025_10_genomovars_not_in_r2025_09.tsv', separator='\t')

# extract fasta sequences for r2025_10 genomovars not in r2025_09
# !seqkit grep \
#     /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_genomovars_reps.fna.gz \
#     --pattern-file r2025_10_genomovars_not_in_r2025_09.tsv \
#     -o r2025_10_genomovars_not_in_r2025_09.fna.gz

Number of r2025_10 genomovars: 520987
Number of r2025_09 genomovars: 458321
Number of r2025_10 genomovars not in r2025_09: 64099


In [ ]:
# # split UHVDB genomovars reps into chunks with 1K sequences
# !seqkit split2 \
#     r2025_10_genomovars_not_in_r2025_09.fna.gz \
#     --by-size 1000 \
#     --out-dir ./uhvdb_r2025_10_genomovar_chunks/

In [6]:
# create config file for annotations
import glob

task_id = 459
with open("annotations_r2025_10_config.txt", "w") as f:
    f.write("TaskArrayID\tFastA\tName\n")
    for file in glob.glob("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_4/uhvdb_r2025_10_genomovar_chunks/*.fna.gz"):
            f.write(f"{task_id}\t{file}\t{file.split('/')[-1].rsplit('.fna')[0]}\n")
            task_id += 1
f.close()